# 7.4 安全对齐 (Safety Alignment)

安全对齐确保模型不会生成有害、偏见或危险的内容，是模型部署前的关键步骤。

本节涵盖：
- 红队测试
- 安全微调
- 宪法AI（Constitutional AI）

## 1. 红队测试

**目的**：主动发现模型的安全漏洞

**基本原理**：通过系统性地构造对抗性输入，测试模型是否会生成有害内容，发现安全漏洞后用于改进模型。

**红队测试方法**：
- 人工红队：安全专家手动构造攻击提示
- 自动红队：使用另一个模型自动生成攻击提示
- 梯度攻击：基于梯度的方法寻找最可能触发有害输出的输入

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class SafetyClassifier(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 128), nn.ReLU(),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.net(x)

classifier = SafetyClassifier()
safe_prompts = torch.randn(4, 64)
unsafe_prompts = torch.randn(4, 64) + 0.5

safe_logits = classifier(safe_prompts)
unsafe_logits = classifier(unsafe_prompts)

safe_preds = safe_logits.argmax(dim=-1)
unsafe_preds = unsafe_logits.argmax(dim=-1)

print('=== Red Teaming ===')
print(f'Safe prompt predictions: {safe_preds.tolist()} (0=safe, 1=unsafe)')
print(f'Unsafe prompt predictions: {unsafe_preds.tolist()}')
print(f'\nRed team finds inputs where the model fails to detect unsafe content.')
print(f'These failures become training data for safety improvement.')

## 2. 安全微调

**目的**：在安全数据上微调模型

**基本原理**：收集安全相关的指令-回复对，训练模型拒绝有害请求、提供安全回复。

**安全微调数据**：
- 有害请求 + 拒绝回复
- 边界请求 + 谨慎回复
- 正常请求 + 正常回复

**挑战**：过度安全（over-refusal）——模型拒绝本应回答的正常问题

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

safety_categories = {
    'violence': {'weight': 1.0, 'n_examples': 500},
    'hate_speech': {'weight': 1.0, 'n_examples': 500},
    'self_harm': {'weight': 1.0, 'n_examples': 300},
    'sexual': {'weight': 0.8, 'n_examples': 400},
    'misinformation': {'weight': 0.7, 'n_examples': 300},
    'borderline': {'weight': 0.5, 'n_examples': 200},
}

print('=== Safety Fine-Tuning ===')
print(f'Safety categories and data allocation:')
total = sum(c['n_examples'] for c in safety_categories.values())
for cat, info in safety_categories.items():
    print(f'  {cat:15s}: {info["n_examples"]:>4d} examples (weight={info["weight"]})')
print(f'  Total: {total} examples')

print(f'\nOver-refusal problem:')
print(f'  "How to cook chicken?" -> "I cannot help with cooking." (WRONG)')
print(f'  "How to make a weapon?" -> "I cannot help with that." (CORRECT)')
print(f'\nKey: Balance safety with helpfulness to avoid over-refusal.')

## 3. 宪法AI（Constitutional AI）

**目的**：通过AI自我批评和修正实现安全对齐

**基本原理**：定义一组"宪法原则"，让AI根据这些原则对自己的输出进行批评和修正，迭代改进直到输出符合所有原则。

**流程**：
1. 模型生成初始回复（可能有害）
2. 根据宪法原则进行自我批评
3. 根据批评修正回复
4. 用修正后的数据训练模型

**代表**：Anthropic Claude系列

In [ ]:
import torch

torch.manual_seed(42)

constitution_principles = [
    'Choose the response that is most helpful and least harmful',
    'Choose the response that is most honest and truthful',
    'Choose the response that is least offensive and controversial',
    'Choose the response that respects human autonomy and dignity',
    'Choose the response that does not facilitate harmful actions',
]

print('=== Constitutional AI ===')
print('Constitutional principles:')
for i, principle in enumerate(constitution_principles, 1):
    print(f'  {i}. {principle}')

print(f'\nSelf-correction process:')
print(f'  Step 1: Model generates initial response (may be harmful)')
print(f'  Step 2: Model critiques response against each principle')
print(f'  Step 3: Model revises response based on critique')
print(f'  Step 4: Revised responses become training data')
print(f'\nKey: Constitutional AI enables scalable safety alignment')
print(f'without requiring large-scale human annotation.')